# 05 — Summary Report Generator

Reads outputs from notebooks 01–04 and generates:
- `reports/REPORT.md` — human-readable markdown summary
- `data/processed/incident_analysis_report.xlsx` — Excel export with one sheet per dataset

**Learning notes:**
- This notebook produces a *reproducible* report. Every time you re-run it after new data, the report updates automatically — you are not hand-writing it.
- This is the integration point. It reads what the other notebooks wrote. If a prior notebook hasn't been run, this will fail with a helpful message telling you which file is missing.

**Prerequisites:** Run notebooks 01–04 in order before this one.

In [ ]:
import sys
import os
sys.path.insert(0, "..")

import pandas as pd
from src.helpers import load_data

# Guard: ensure prior notebooks have been run
required_files = [
    "../data/processed/summary_stats.csv",
    "../data/processed/control_coverage.csv",
    "../data/processed/team_risk_scores.csv",
    "../data/processed/theme_frequencies.csv",
]
for fpath in required_files:
    assert os.path.exists(fpath), (
        f"Missing: {fpath}\n"
        "Run notebooks 01–04 first to generate processed outputs."
    )

print("All required processed files found.")

In [ ]:
# Load raw data
data = load_data("../data/raw")
incidents = data["incidents"]
ris       = data["ris"]
controls  = data["controls"]
mappings  = data["mappings"]

# Load processed outputs from prior notebooks
summary_stats = pd.read_csv("../data/processed/summary_stats.csv")
control_cov   = pd.read_csv("../data/processed/control_coverage.csv")
team_risk     = pd.read_csv("../data/processed/team_risk_scores.csv", index_col=0)
theme_freq    = pd.read_csv("../data/processed/theme_frequencies.csv")

print("All data loaded.")

## 1. Extract Key Facts for the Report

In [ ]:
def get_stat(df, metric):
    return int(df.loc[df["metric"] == metric, "value"].iloc[0])

total_incidents = get_stat(summary_stats, "Total Incidents")
open_incidents  = get_stat(summary_stats, "Open Incidents")
total_ris       = get_stat(summary_stats, "Total RIs")
open_ris        = get_stat(summary_stats, "Open RIs")

zero_cov_controls = control_cov[control_cov["ri_count"] == 0]
top_risk_team     = team_risk.index[0]
top_risk_score    = int(team_risk.iloc[0]["total_risk_score"])
top_themes        = theme_freq["word"].head(5).tolist()

print(f"Total incidents    : {total_incidents}")
print(f"Open incidents     : {open_incidents}")
print(f"Total RIs          : {total_ris}")
print(f"Open RIs           : {open_ris}")
print(f"Zero-cov controls  : {len(zero_cov_controls)}")
print(f"Top risk team      : {top_risk_team} (score: {top_risk_score})")
print(f"Top themes         : {top_themes}")

## 2. Generate REPORT.md

In [ ]:
os.makedirs("../reports", exist_ok=True)

lines = [
    "# Incident Analysis — Summary Report",
    "",
    f"_Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}_",
    "",
    "---",
    "",
    "## 1. Dataset Overview",
    "",
    "| Metric | Value |",
    "|--------|-------|",
    f"| Total Incidents | {total_incidents} |",
    f"| Open Incidents | {open_incidents} |",
    f"| Total Remediation Items | {total_ris} |",
    f"| Open Remediation Items | {open_ris} |",
    f"| Controls Defined | {len(controls)} |",
    f"| Control-RI Mappings | {len(mappings)} |",
    "",
    "---",
    "",
    "## 2. Control Coverage Gaps",
    "",
    f"{len(zero_cov_controls)} controls have **zero RI coverage** (no remediation items mapped to them):",
    "",
]

for _, row in zero_cov_controls.iterrows():
    lines.append(
        f"- `{row['control_id']}` — {row['control_name']} _(Domain: {row['control_domain']})_"
    )

lines += [
    "",
    "**Recommendation:** Review these controls. Either create RIs to address them, "
    "or formally accept/retire the control.",
    "",
    "---",
    "",
    "## 3. Team Risk Summary",
    "",
    f"The highest-risk team is **{top_risk_team}** with a total risk score of **{top_risk_score}**.",
    "",
    "Top 5 teams by risk score:",
    "",
    "| Rank | Team | Risk Score |",
    "|------|------|------------|",
]

for rank, (team, row) in enumerate(team_risk.head(5).iterrows(), start=1):
    lines.append(f"| {rank} | {team} | {int(row['total_risk_score'])} |")

lines += [
    "",
    "---",
    "",
    "## 4. Recurring Themes",
    "",
    "Most frequently occurring terms across incident titles and descriptions:",
    "",
    f"**Top 5 themes:** {', '.join(top_themes)}",
    "",
    "These themes may indicate systemic issues that controls or processes are not fully addressing.",
    "",
    "---",
    "",
    "## 5. Recommendations",
    "",
    "1. Address zero-coverage controls by creating targeted remediation items.",
    "2. Prioritise the top 3 risk teams for immediate review and resource allocation.",
    "3. Investigate recurring themes — they may signal process failures or training gaps.",
    "4. Re-run this analysis monthly to track improvement in RI closure rates.",
    "",
    "---",
    "",
    "_Report auto-generated by `notebooks/05_summary_report.ipynb`_",
]

report_text = "\n".join(lines)
with open("../reports/REPORT.md", "w") as f:
    f.write(report_text)

print("REPORT.md written to ../reports/REPORT.md")

## 3. Export Excel Workbook

In [ ]:
excel_path = "../data/processed/incident_analysis_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    incidents.to_excel(writer,   sheet_name="Incidents",        index=False)
    ris.to_excel(writer,         sheet_name="RemediationItems",  index=False)
    controls.to_excel(writer,    sheet_name="Controls",          index=False)
    mappings.to_excel(writer,    sheet_name="Mappings",          index=False)
    team_risk.to_excel(writer,   sheet_name="TeamRiskScores")
    control_cov.to_excel(writer, sheet_name="ControlCoverage",   index=False)
    theme_freq.to_excel(writer,  sheet_name="ThemeFrequencies",  index=False)

print(f"Excel report saved to: {excel_path}")

## 4. Final Summary

In [ ]:
print("=" * 50)
print("Summary Report Complete")
print("=" * 50)
print(f"  REPORT.md         : ../reports/REPORT.md")
print(f"  Excel workbook    : {excel_path}")
print()
print("Key findings:")
print(f"  - {len(zero_cov_controls)} controls with zero RI coverage")
print(f"  - Highest risk team: {top_risk_team} (score: {top_risk_score})")
print(f"  - Top themes: {', '.join(top_themes)}")
print("=" * 50)